# PAD-UFES-20: Metadata Exploration

This notebook contains metadata-only exploration, not model performance or fairness results. The input must match the SHA-256 displayed on the [official file page](https://data.mendeley.com/datasets/zr7vgbcyr2/1/files/fa850265-57da-48f0-ba3e-998b3e44b1f6).

No patient rows, patient identifiers or medical images are displayed. See the [dataset card](../reports/dataset_card.md) and [interpretation report](../reports/eda_report.md).

In [1]:
from pathlib import Path
import json
import os
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "configs/study.json").exists():
    ROOT = ROOT.parent
assert (ROOT / "configs/study.json").exists(), "Run from the repository or notebooks directory."
sys.path.insert(0, str(ROOT))
os.environ["MPLCONFIGDIR"] = str(ROOT / "work/matplotlib")
from src.data.metadata_audit import read_metadata, sha256_file, audit_metadata

source = ROOT / "data/raw/metadata.csv"
frame = read_metadata(source)
audit = audit_metadata(frame, source_sha256=sha256_file(source))
print("Official SHA-256 verified:", audit["source_sha256"])
pd.DataFrame([audit["counts"]])

Official SHA-256 verified: 14d145235cedb022548257acb0d84dcd949e2c916f65d2baa7c38ed5339e9527


,image_records,metadata_columns,unique_patients,unique_lesion_id_strings,patient_lesion_pairs
0,2298,26,1373,1641,1891


## Correct analysis units

Image rows are not independent patients. Raw lesion IDs are not globally unique. The patient–lesion pair is the provisional aggregation key, and patients remain the grouping unit for later data separation and statistical resampling.

In [2]:
pd.DataFrame(audit["class_support"]).T.rename_axis("diagnosis")

,images,patient_lesion_pairs,patients
diagnosis,,,
ACK,730,644,525
BCC,845,652,513
MEL,52,36,36
NEV,244,216,185
SCC,192,145,133
SEK,235,198,176


## Demographic coverage and consistency

Patient summaries count each patient once. A single valid observed value summarises that patient; conflicting values remain unresolved. This operation does not fill or overwrite source rows. Recorded gender is not automatically verified biological sex, and Fitzpatrick type is not an objective skin-colour measurement.

In [3]:
pd.DataFrame({name: values["status_patients"]
              for name, values in audit["patient_demographics"].items()}).T

,known,missing,conflict,invalid
age,1283,0,90,0
gender,794,579,0,0
fitspatrick,794,579,0,0


In [4]:
print("Age groups:")
print(json.dumps(audit["patient_demographics"]["age"]["age_group_patients"], indent=2))
print("Recorded gender, known patients:")
print(json.dumps(audit["patient_demographics"]["gender"]["known_patient_distribution"], indent=2))
print("Recorded Fitzpatrick type, with small counts suppressed:")
print(json.dumps(audit["patient_demographics"]["fitspatrick"]["known_patient_distribution"], indent=2))

Age groups:
{
  "Under 60": 608,
  "60 and over": 675,
  "Unresolved / missing": 90
}
Recorded gender, known patients:
{
  "FEMALE": 416,
  "MALE": 378
}
Recorded Fitzpatrick type, with small counts suppressed:
{
  "1": 74,
  "2": 465,
  "3": 212,
  "4": 34,
  "5": "<10",
  "6": "<10"
}


## Missing metadata

Blank and literal UNK values are counted separately. Record-level percentages use image rows as the denominator. They must not be labelled patient percentages.

In [5]:
pd.DataFrame(audit["missingness_by_image_record"]).T.sort_values("unavailable_rows", ascending=False)

,blank_rows,unknown_rows,unavailable_rows,unavailable_percent
background_father,818.0,95.0,913.0,39.730
background_mother,822.0,84.0,906.0,39.426
fitspatrick,804.0,0.0,804.0,34.987
skin_cancer_history,804.0,0.0,804.0,34.987
diameter_2,804.0,0.0,804.0,34.987
diameter_1,804.0,0.0,804.0,34.987
has_sewage_system,804.0,0.0,804.0,34.987
has_piped_water,804.0,0.0,804.0,34.987
cancer_history,804.0,0.0,804.0,34.987
gender,804.0,0.0,804.0,34.987


In [6]:
pd.DataFrame(audit["demographic_coverage_by_diagnosis"]["gender"]).T.rename_axis("diagnosis")

,known_images,missing_images
diagnosis,,
ACK,283,447
BCC,845,0
MEL,52,0
NEV,75,169
SCC,192,0
SEK,47,188


## Integrity checks

These checks concern metadata and filename conventions only. They do not prove that actual images are readable, unique or matched correctly on disk.

In [7]:
print(json.dumps(audit["integrity_checks"], indent=2))

{
  "duplicate_image_ids_extra_rows": 0,
  "duplicate_full_rows": 0,
  "lesion_ids_used_by_multiple_patients": 250,
  "patient_lesion_attribute_conflicts": {
    "diagnostic": 0,
    "age": 0,
    "gender": 0,
    "fitspatrick": 0,
    "biopsed": 0
  },
  "filename_identifier_mismatches": 0
}


## Aggregate figures

Run `python scripts/audit_data.py` from the repository root to regenerate all six figures as SVG and PNG.

![Class support](../figures/eda/class_support.png)

![Patient demographics](../figures/eda/patient_demographics.png)

![Patient age distribution](../figures/eda/patient_age_distribution.png)

![Fitzpatrick coverage](../figures/eda/fitzpatrick_coverage.png)

![Metadata missingness](../figures/eda/metadata_missingness.png)

![Missingness by diagnosis](../figures/eda/demographic_missingness_by_diagnosis.png)

## Interpretation and next stage

- Keep raw lesion IDs separate across patients.
- Do not silently resolve conflicting ages.
- Keep missing-demographic records for overall image-only modelling and report audit coverage.
- Treat rare diagnoses and demographic categories cautiously.
- Image acquisition, content checks, patient partitions, models and calibration remain pending.

This notebook makes no claim about predictive accuracy, model fairness or calibration because models have not been trained.